## OCA Indonesia - Virtual Internship Project 2: Active User Behavior & Segmentation
## RFM Segmentation Script (FM-based, R excluded — semua user aktif di period yang sama)

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

CONFIG

In [11]:
INPUT_FILE  = "OCA Segmentation Unified Table.csv"   # ganti path jika perlu
OUTPUT_CSV  = "oca_user_segments.csv"

SEGMENT_COLORS = {
    "Power User"  : "#1a6faf",
    "Active User" : "#43aa8b",
    "Regular User": "#f4a261",
}


1. Load Data

In [12]:
df = pd.read_csv(INPUT_FILE)

In [13]:
# Konversi tipe
num_cols = [
    "wa_total_transactions", "wa_charged_transactions", "wa_total_revenue",
    "wa_active_days", "wa_delivered", "wa_read", "wa_rejected", "wa_submitted",
    "sms_total_transactions", "sms_charged_transactions", "sms_total_revenue",
    "sms_active_days", "sms_sent", "sms_failed", "sms_total_messages",
    "email_total_transactions", "email_charged_transactions", "email_total_revenue",
    "email_active_days", "email_delivered", "email_opened", "email_clicked",
    "email_processed", "email_failed",
    "call_total_transactions", "call_charged_transactions", "call_total_revenue",
    "call_active_days", "call_answered", "call_rna", "call_failed",
    "total_transactions", "total_revenue", "total_active_days", "channel_count",
]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

print(f"Data loaded: {len(df)} users\n")

Data loaded: 20 users



2. Feature Engineering

In [14]:
# Revenue share per channel
df["wa_rev_pct"]    = df["wa_total_revenue"]    / df["total_revenue"] * 100
df["sms_rev_pct"]   = df["sms_total_revenue"]   / df["total_revenue"] * 100
df["email_rev_pct"] = df["email_total_revenue"]  / df["total_revenue"] * 100
df["call_rev_pct"]  = df["call_total_revenue"]   / df["total_revenue"] * 100

# Dominant channel (by revenue)
df["dominant_channel"] = df[
    ["wa_rev_pct", "sms_rev_pct", "email_rev_pct", "call_rev_pct"]
].idxmax(axis=1).str.replace("_rev_pct", "").str.upper()
df["dominant_channel"] = df["dominant_channel"].replace({
    "WA": "WhatsApp", "SMS": "SMS", "EMAIL": "Email", "CALL": "Call"
})

# Avg revenue per transaction
df["rev_per_tx"] = df["total_revenue"] / df["total_transactions"]

# WA delivery rate (charged / total)
df["wa_delivery_rate"] = df["wa_charged_transactions"] / df["wa_total_transactions"] * 100

# Call answer rate
df["call_answer_rate"] = df["call_answered"] / df["call_total_transactions"] * 100

3. RFM Scoring

In [18]:
# R: Recency diabaikan — semua user last_active = 0025-03-31 (recency = 0)
# F: Frequency → total_transactions
# M: Monetary  → total_revenue

# Skor 1–3 menggunakan percentile dari data aktual
def score_col(series, ascending=True):
    """Bagi jadi 3 bucket berdasarkan percentile."""
    p33 = series.quantile(0.33)
    p66 = series.quantile(0.66)
    if ascending:   # lebih tinggi = lebih baik
        return series.apply(lambda x: 3 if x >= p66 else (2 if x >= p33 else 1))
    else:           # lebih rendah = lebih baik (untuk recency)
        return series.apply(lambda x: 3 if x <= p33 else (2 if x <= p66 else 1))

df["f_score"] = score_col(df["total_transactions"], ascending=True)
df["m_score"] = score_col(df["total_revenue"],      ascending=True)
df["fm_total"] = df["f_score"] + df["m_score"]

print("=== F Score distribution ===")
print(df["f_score"].value_counts().sort_index())
print(f"\nF thresholds — P33: {df['total_transactions'].quantile(0.33):,.0f}  |  P66: {df['total_transactions'].quantile(0.66):,.0f}")

print("\n=== M Score distribution ===")
print(df["m_score"].value_counts().sort_index())
print(f"\nM thresholds — P33: {df['total_revenue'].quantile(0.33):,.0f}  |  P66: {df['total_revenue'].quantile(0.66):,.0f}")


=== F Score distribution ===
f_score
1    7
2    6
3    7
Name: count, dtype: int64

F thresholds — P33: 1,536  |  P66: 6,068

=== M Score distribution ===
m_score
1    7
2    6
3    7
Name: count, dtype: int64

M thresholds — P33: 336,028  |  P66: 1,280,687


4. Rule Based Segmentation

In [19]:
# Catatan: mono-channel & dormant tidak ditemukan dalam dataset ini
def assign_segment(row):
    if row["fm_total"] >= 6:       # F=3 + M=3
        return "Power User"
    elif row["fm_total"] >= 4:     # kombinasi menengah
        return "Active User"
    else:                          # FM rendah
        return "Regular User"

df["segment"] = df.apply(assign_segment, axis=1)

print("\n=== Segment distribution ===")
print(df["segment"].value_counts())


=== Segment distribution ===
segment
Active User     7
Regular User    7
Power User      6
Name: count, dtype: int64


5. Export CSV

In [20]:
cols_export = [
    "user_id", "user_name", "field_of_business", "join_date",
    "total_transactions", "total_revenue", "total_active_days",
    "wa_total_transactions", "sms_total_transactions",
    "email_total_transactions", "call_total_transactions",
    "wa_total_revenue", "sms_total_revenue",
    "email_total_revenue", "call_total_revenue",
    "wa_rev_pct", "sms_rev_pct", "email_rev_pct", "call_rev_pct",
    "dominant_channel", "rev_per_tx",
    "wa_delivery_rate", "call_answer_rate",
    "f_score", "m_score", "fm_total", "segment",
]
df[cols_export].to_csv(OUTPUT_CSV, index=False)
print(f"\nSegment table saved → {OUTPUT_CSV}")


Segment table saved → oca_user_segments.csv


In [24]:
# VISUALISASI
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"  : "sans-serif",
    "font.size"    : 10,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

colors = df["segment"].map(SEGMENT_COLORS)

# ── VIZ 1: Scatter Plot — Revenue vs Transactions (colored by segment) ────────
fig, ax = plt.subplots(figsize=(11, 7))

for seg, grp in df.groupby("segment"):
    ax.scatter(
        grp["total_transactions"], grp["total_revenue"],
        color=SEGMENT_COLORS[seg], s=160, alpha=0.85,
        edgecolors="white", linewidths=0.8, label=seg, zorder=3
    )

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"Rp{x/1e6:.1f}M"))
ax.set_xlabel("Total Transactions", fontsize=11)
ax.set_ylabel("Total Revenue", fontsize=11)
ax.set_title("User Segmentation: Revenue vs Transaction Volume\n(OCA Indonesia — FM-based RFM)", fontsize=13, fontweight="bold", pad=14)
ax.legend(title="Segment", fontsize=9, title_fontsize=9)
ax.grid(axis="both", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig("viz1_scatter_segment.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz1_scatter_segment.png")

# ── VIZ 2: Revenue Breakdown per User (stacked bar, sorted by total) ─────────
df_sorted = df.sort_values("total_revenue", ascending=True)
y_pos     = range(len(df_sorted))
bar_h     = 0.65

fig, ax = plt.subplots(figsize=(12, 8))

left = np.zeros(len(df_sorted))
channel_data = {
    "WhatsApp": df_sorted["wa_total_revenue"].values,
    "SMS"      : df_sorted["sms_total_revenue"].values,
    "Email"    : df_sorted["email_total_revenue"].values,
    "Call"     : df_sorted["call_total_revenue"].values,
}
ch_colors = {"WhatsApp": "#25D366", "SMS": "#4a90d9", "Email": "#f4a261", "Call": "#e63946"}

for ch, vals in channel_data.items():
    bars = ax.barh(list(y_pos), vals / 1e6, left=left / 1e6,
                   height=bar_h, color=ch_colors[ch], label=ch, alpha=0.88)
    left += vals

# Segment label di sisi kanan
for i, (_, row) in enumerate(df_sorted.iterrows()):
    seg_short = {"Power User": "⭐ Power", "Active User": "✦ Active", "Regular User": "● Regular"}[row["segment"]]
    ax.text(row["total_revenue"] / 1e6 + 0.05, i, seg_short,
            va="center", fontsize=7.5, color=SEGMENT_COLORS[row["segment"]], fontweight="bold")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(df_sorted["user_name"].str.replace("PT |CV ", "", regex=True), fontsize=8.5)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"Rp{x:.1f}M"))
ax.set_xlabel("Total Revenue (Millions)", fontsize=11)
ax.set_title("Revenue by User & Channel\n(Sorted by Total Revenue)", fontsize=13, fontweight="bold", pad=14)
ax.legend(title="Channel", loc="lower right", fontsize=9)
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("viz2_revenue_breakdown.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz2_revenue_breakdown.png")

# ── VIZ 3: Segment Summary — Avg Metrics per Segment ─────────────────────────
seg_summary = df.groupby("segment").agg(
    user_count       = ("user_id", "count"),
    avg_revenue      = ("total_revenue", "mean"),
    total_revenue    = ("total_revenue", "sum"),
    avg_transactions = ("total_transactions", "mean"),
    avg_active_days  = ("total_active_days", "mean"),
    avg_wa_pct       = ("wa_rev_pct", "mean"),
    avg_sms_pct      = ("sms_rev_pct", "mean"),
    avg_email_pct    = ("email_rev_pct", "mean"),
    avg_call_pct     = ("call_rev_pct", "mean"),
).reindex(["Power User", "Active User", "Regular User"])

fig = plt.figure(figsize=(16, 6))
# 4 kolom: chart1 | chart2 | legend | chart3
gs  = GridSpec(1, 4, figure=fig, wspace=0.45, width_ratios=[1, 1, 0.35, 1])

# 3a: Avg Revenue per Segment
ax1 = fig.add_subplot(gs[0])
segs = seg_summary.index.tolist()
avgs = seg_summary["avg_revenue"].values / 1e6
bars = ax1.bar(segs, avgs, color=[SEGMENT_COLORS[s] for s in segs], width=0.55, alpha=0.88)
for bar, val in zip(bars, avgs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"Rp{val:.2f}M", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.set_ylabel("Avg Revenue (Millions)")
ax1.set_title("Avg Revenue\nper Segment", fontweight="bold")
ax1.set_xticklabels(segs, rotation=12, ha="right", fontsize=9)
ax1.grid(axis="y", linestyle="--", alpha=0.35)

# 3b: Avg Transactions per Segment
ax2 = fig.add_subplot(gs[1])
avgt = seg_summary["avg_transactions"].values
bars2 = ax2.bar(segs, avgt, color=[SEGMENT_COLORS[s] for s in segs], width=0.55, alpha=0.88)
for bar, val in zip(bars2, avgt):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_ylabel("Avg Transactions")
ax2.set_title("Avg Transaction Volume\nper Segment", fontweight="bold")
ax2.set_xticklabels(segs, rotation=12, ha="right", fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax2.grid(axis="y", linestyle="--", alpha=0.35)

# 3b_legend: legend di kolom tengah (gs[2])
ax_legend = fig.add_subplot(gs[2])
ax_legend.axis("off")
legend_patches = [mpatches.Patch(color=col, label=ch) for ch, col in ch_colors.items()]
ax_legend.legend(
    handles=legend_patches, title="Channel", fontsize=10,
    title_fontsize=10, loc="center", frameon=True,
    borderpad=1.2, labelspacing=1.0,
)

# 3c: Channel Revenue Mix per Segment (stacked bar)
ax3 = fig.add_subplot(gs[3])
x   = np.arange(len(segs))
w   = 0.55
bot = np.zeros(len(segs))
for ch, col in ch_colors.items():
    key  = f"avg_{ch.lower()}_pct" if ch != "WhatsApp" else "avg_wa_pct"
    vals = seg_summary[key].values
    ax3.bar(x, vals, bottom=bot, width=w, color=col, label=ch, alpha=0.88)
    for i, (v, b) in enumerate(zip(vals, bot)):
        if v > 3:
            ax3.text(x[i], b + v/2, f"{v:.0f}%", ha="center", va="center",
                     fontsize=8, color="white", fontweight="bold")
    bot += vals
ax3.set_xticks(x)
ax3.set_xticklabels(segs, rotation=12, ha="right", fontsize=9)
ax3.set_ylabel("Revenue Share (%)")
ax3.set_title("Channel Revenue Mix\nper Segment", fontweight="bold")
ax3.set_ylim(0, 110)
ax3.grid(axis="y", linestyle="--", alpha=0.35)

fig.suptitle("Segment Summary — OCA Indonesia", fontsize=14, fontweight="bold", y=1.02)
plt.savefig("viz3_segment_summary.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz3_segment_summary.png")

# ── VIZ 4: User Persona Table ─────────────────────────────────────────────────
persona_data = {
    "Power User": {
        "users"        : df[df.segment=="Power User"]["user_name"].str.replace("PT |CV ","",regex=True).tolist(),
        "description"  : "Enterprise-scale clients. Daily high-volume blast across all channels. SMS & WA drive >90% of revenue.",
        "rev_range"    : f"Rp{df[df.segment=='Power User']['total_revenue'].min()/1e6:.1f}M – Rp{df[df.segment=='Power User']['total_revenue'].max()/1e6:.1f}M",
        "avg_tx"       : f"{df[df.segment=='Power User']['total_transactions'].mean():,.0f}",
        "top_channel"  : "SMS (~45%) + WA (~40%)",
        "action"       : "Loyalty program, dedicated account manager, early access to new features",
    },
    "Active User": {
        "users"        : df[df.segment=="Active User"]["user_name"].str.replace("PT |CV ","",regex=True).tolist(),
        "description"  : "Mid-size clients. Consistent multi-channel usage. SMS still dominant but WA growing.",
        "rev_range"    : f"Rp{df[df.segment=='Active User']['total_revenue'].min()/1e6:.1f}M – Rp{df[df.segment=='Active User']['total_revenue'].max()/1e6:.1f}M",
        "avg_tx"       : f"{df[df.segment=='Active User']['total_transactions'].mean():,.0f}",
        "top_channel"  : "SMS (~46%) + WA (~39%)",
        "action"       : "Upsell to higher volume package, introduce automation & scheduled blast features",
    },
    "Regular User": {
        "users"        : df[df.segment=="Regular User"]["user_name"].str.replace("PT |CV ","",regex=True).tolist(),
        "description"  : "Smaller clients. Lower blast volume. Call answer rate low (RNA high) — potential to shift to WA/SMS.",
        "rev_range"    : f"Rp{df[df.segment=='Regular User']['total_revenue'].min()/1e6:.1f}M – Rp{df[df.segment=='Regular User']['total_revenue'].max()/1e6:.1f}M",
        "avg_tx"       : f"{df[df.segment=='Regular User']['total_transactions'].mean():,.0f}",
        "top_channel"  : "SMS (~46%) + WA (~40%)",
        "action"       : "Offer volume growth incentive, replace Call with WhatsApp to reduce RNA rate",
    },
}

fig, ax = plt.subplots(figsize=(16, 6))
ax.axis("off")

headers = ["Segment", "Users", "Revenue Range", "Avg Tx", "Top Channel", "Description", "Recommended Action"]
rows_table = []
for seg, info in persona_data.items():
    rows_table.append([
        seg,
        "\n".join(info["users"]),
        info["rev_range"],
        info["avg_tx"],
        info["top_channel"],
        info["description"],
        info["action"],
    ])

col_widths = [0.10, 0.14, 0.10, 0.07, 0.12, 0.25, 0.22]
table = ax.table(
    cellText=rows_table,
    colLabels=headers,
    cellLoc="left",
    loc="center",
    colWidths=col_widths,
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 3.8)

# Header styling
for j in range(len(headers)):
    table[0, j].set_facecolor("#2d3142")
    table[0, j].set_text_props(color="white", fontweight="bold")

# Row coloring per segment
seg_rows = {"Power User": 1, "Active User": 2, "Regular User": 3}
for seg, row_idx in seg_rows.items():
    base_color = SEGMENT_COLORS[seg]
    for j in range(len(headers)):
        table[row_idx, j].set_facecolor(base_color + "22")  # light tint
        if j == 0:
            table[row_idx, j].set_text_props(color=SEGMENT_COLORS[seg], fontweight="bold")

ax.set_title("User Persona Table — OCA Indonesia", fontsize=14, fontweight="bold", pad=16)
plt.tight_layout()
plt.savefig("viz4_persona_table.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz4_persona_table.png")


Saved: viz1_scatter_segment.png
Saved: viz2_revenue_breakdown.png
Saved: viz3_segment_summary.png
Saved: viz4_persona_table.png


6. PRINT SEGMENT SUMMARY TABLE

In [25]:
print("\n" + "="*65)
print("SEGMENT SUMMARY")
print("="*65)
print(f"{'Segment':<14} {'Users':>6} {'Avg Revenue':>14} {'Avg Tx':>9} {'FM Score':>9}")
print("-"*65)
for seg in ["Power User", "Active User", "Regular User"]:
    g = df[df.segment == seg]
    print(f"{seg:<14} {len(g):>6} {g['total_revenue'].mean():>14,.0f} {g['total_transactions'].mean():>9,.0f} {g['fm_total'].mean():>9.1f}")
print("="*65)

print("\n✅ Done. Files generated:")
print("   oca_user_segments.csv")
print("   viz1_scatter_segment.png")
print("   viz2_revenue_breakdown.png")
print("   viz3_segment_summary.png")
print("   viz4_persona_table.png")


SEGMENT SUMMARY
Segment         Users    Avg Revenue    Avg Tx  FM Score
-----------------------------------------------------------------
Power User          6      3,242,447    15,349       6.0
Active User         7        610,473     2,867       4.3
Regular User        7        316,919     1,512       2.0

✅ Done. Files generated:
   oca_user_segments.csv
   viz1_scatter_segment.png
   viz2_revenue_breakdown.png
   viz3_segment_summary.png
   viz4_persona_table.png


Time Trends test

In [27]:
"""7. Hourly Activity Analysis"""

# pd.to_datetime tidak bisa handle tahun 0025 (OutOfBoundsDatetime)
# gunakan string extraction untuk hour
df_raw = pd.read_csv("OCA Union Table.csv", low_memory=False)
df_raw["hour"] = df_raw["created_at"].str[11:13].astype(int)
df_raw = df_raw.merge(df[["user_id", "segment"]], on="user_id", how="left")

# VIZ 5: Hourly Transaction Pattern per Segment
hourly = df_raw.groupby(["segment", "hour"]).size().reset_index(name="tx_count")
seg_totals = hourly.groupby("segment")["tx_count"].transform("sum")
hourly["tx_pct"] = hourly["tx_count"] / seg_totals * 100

fig, ax = plt.subplots(figsize=(12, 6))
for seg in ["Power User", "Active User", "Regular User"]:
    data = hourly[hourly["segment"] == seg].sort_values("hour")
    ax.plot(data["hour"], data["tx_pct"], color=SEGMENT_COLORS[seg],
            linewidth=2.5, marker="o", markersize=5, label=seg)
ax.axvspan(8, 17, alpha=0.07, color="gray", label="Business Hours (08-17)")
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Hour of Day", fontsize=11)
ax.set_ylabel("% of Daily Transactions", fontsize=11)
ax.set_title("Hourly Transaction Pattern per Segment\n(OCA Indonesia)", fontsize=13, fontweight="bold", pad=14)
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig("viz5_hourly_pattern.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz5_hourly_pattern.png")

# VIZ 6: Heatmap Hour x Channel per Segment
channels = ["Whatsapp", "SMS", "Email", "Call"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax_i, seg in zip(axes, ["Power User", "Active User", "Regular User"]):
    heat = df_raw[df_raw["segment"]==seg].groupby(["channel","hour"]).size().unstack(fill_value=0)
    heat = heat.reindex(index=channels, columns=range(24), fill_value=0)
    im = ax_i.imshow(heat.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")
    ax_i.set_xticks(range(0, 24, 2))
    ax_i.set_xticklabels([f"{h:02d}" for h in range(0, 24, 2)], fontsize=8)
    ax_i.set_yticks(range(len(channels)))
    ax_i.set_yticklabels(channels, fontsize=9)
    ax_i.set_title(seg, fontweight="bold", fontsize=11, color=SEGMENT_COLORS[seg])
    ax_i.set_xlabel("Hour of Day", fontsize=9)
    plt.colorbar(im, ax=ax_i, shrink=0.8, label="Tx Count")
fig.suptitle("Transaction Heatmap: Hour x Channel per Segment", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz6_heatmap_hour_channel.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: viz6_heatmap_hour_channel.png")

Saved: viz5_hourly_pattern.png
Saved: viz6_heatmap_hour_channel.png
